# Visualisasi Artefak PCA — PDAM Pump Sentinel

Notebook ini mendemonstrasikan proses pelatihan **PCA T²/Q** dan visualisasi artefaknya.
Seluruh contoh menggunakan dataset surrogate **SKAB** (water circulation testbed) dalam skala minimal;
**bukan data operasional PDAM nyata**. Tujuannya adalah untuk pembelajaran akademik dan verifikasi pipeline.


## Dataset Demo

Kita menggunakan `tests/fixtures/skab_tiny.csv` — subset kecil dari SKAB dengan 3 baris telemetry.
File ini memastikan notebook dapat dijalankan secara mandiri tanpa akses ke dataset lengkap.


In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        fixture = candidate / 'tests' / 'fixtures' / 'skab_tiny.csv'
        if (candidate / 'pyproject.toml').exists() and fixture.exists():
            return candidate
    raise FileNotFoundError('Project root with tests/fixtures/skab_tiny.csv was not found')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INPUT_CSV = PROJECT_ROOT / 'tests' / 'fixtures' / 'skab_tiny.csv'
ARTIFACT_DIR = Path('/tmp/pdam-pump-sentinel-pca-demo/artifacts')

print('Project root:', PROJECT_ROOT)
print('Input CSV :', INPUT_CSV.resolve())
print('Artifact dir:', ARTIFACT_DIR.resolve())


## Pelatihan PCA T²/Q

Konfigurasi pelatihan menggunakan `window_size=1` dan `stride=1` agar cocok dengan dataset mini.
Threshold ditentukan pada kuantil 0.95; MLflow dinonaktifkan agar tidak bergantung pada server tracking.


In [ ]:
from ml.training.train_pca import PcaTrainingConfig, train_pca_from_skab

train_config = PcaTrainingConfig(
    input_path=INPUT_CSV,
    output_dir=ARTIFACT_DIR,
    window_size=1,
    stride=1,
    threshold_quantile=0.95,
    log_mlflow=False,
)

train_result = train_pca_from_skab(train_config)
print('Output dir     :', train_result.output_dir)
print('Artifact paths :', list(train_result.artifact_paths.keys()))
print('Thresholds     :', train_result.thresholds)


## Generasi Visualisasi

Membuat artefak visualisasi (Plotly JSON & HTML) dari hasil pelatihan di atas.


In [ ]:
from ml.visualization.pca_artifacts import (
    PcaArtifactVisualizationConfig,
    generate_pca_artifact_visualizations,
)

viz_config = PcaArtifactVisualizationConfig(
    artifact_dir=ARTIFACT_DIR,
)

viz_artifacts = generate_pca_artifact_visualizations(viz_config)
print('Visualisasi yang dihasilkan:')
for name, path in viz_artifacts.items():
    print(f"  {name}: {path}")


## Ringkasan Hasil

Menampilkan ringkasan metrik dan threshold dalam bentuk tabel pandas.


In [ ]:
import json
import pandas as pd

summary = json.loads(viz_artifacts['summary'].read_text())

df_summary = pd.DataFrame([
    {
        'sample_count': summary['metrics']['sample_count'],
        'anomaly_count': summary['metrics']['anomaly_count'],
        'normal_count': summary['metrics']['normal_count'],
        't2_threshold': summary['thresholds']['t2_threshold'],
        'q_threshold': summary['thresholds']['q_threshold'],
        'artifact_count': summary['artifact_count'],
    }
])

df_summary


## Grafik Scatter T²/Q

Membaca artefak Plotly JSON dan menampilkannya secara offline melalui `IPython.display.HTML`.


In [ ]:
import plotly.io as pio
from IPython.display import HTML

scatter_json = viz_artifacts['scores_scatter_plotly_json']
fig_scatter = pio.read_json(str(scatter_json))
HTML(fig_scatter.to_html(include_plotlyjs=True, full_html=False))


## Grafik Timeline

Menampilkan timeline T², Q, dan score beserta garis threshold.


In [ ]:
timeline_json = viz_artifacts['scores_timeline_plotly_json']
fig_timeline = pio.read_json(str(timeline_json))
HTML(fig_timeline.to_html(include_plotlyjs=True, full_html=False))


## Catatan

- Notebook ini dirancang untuk dijalankan **offline**; Plotly JS disertakan inline pada setiap output HTML.
- Dataset `skab_tiny.csv` hanya berisi 3 baris sehingga hasil visualisasi bersifat ilustratif,
  bukan representasi performa model pada data operasional sesungguhnya.
- Untuk eksperimen lebih lanjut, gunakan dataset SKAB lengkap atau data telemetry pompa PDAM yang valid.
